# Audit compression progress: the five reported numbers

This notebook computes the report defined in *Signed Compression Progress on a Sealed Audit is
Goodhart-Resistant*. Every quantity is in **nats**.

For a declared target distribution `Q` over audit units `z = (x, y)`, the population log loss of a
probability quote `p` is

```
L_Q(p) = mean over audit units of  -log p(y | x)
```

Given a benchmark-owned reference quote `p_ref`, an initial state `p_0` and a terminal state `p_T`,
the report is five numbers.

| # | Number | Definition |
|---|--------|------------|
| 1 | terminal level `B` | `L_Q(p_ref) - L_Q(p_T)` |
| 2 | signed gain `G` | `L_Q(p_0) - L_Q(p_T)` |
| 3 | slack `Gamma` | route-dependent; Section 4 |
| 4 | lower bound on improvement | `G - Gamma` |
| 5 | discrimination ratio | `G / Gamma` |

**What the lower bound licenses.** A run reporting signed gain `G` has improved on the declared
target `Q` by at least `G - Gamma` nats. That statement holds at every horizon and for every
adaptive trajectory covered by the declared event: no term in the number of training steps and no
term in the parameter count enters `Gamma`, so a longer run or a larger model is covered without
change. Because the increments are signed, the reported gain is already net of forgetting.

The statement is about `Q`, under the declared masking policy, in the declared unit. It is not a
forecast of behaviour on other work and carries no claim about safety, values, or coverage outside
`Q`. When `G` does not exceed `Gamma` the lower bound is not positive, and the statement available
is that no improvement is demonstrated at that resolution. The discrimination ratio `G / Gamma` is
dimensionless and comparable across substrates; it exceeds 1 exactly when the lower bound is
positive.

**Vocabulary.** `Gamma` is the *slack*. `G - Gamma` is the *lower bound on improvement*. The object
produced here is a bound.

**Interface.** Every function takes **arrays of per-audit-unit losses**, not model objects: three
numpy vectors of length `n` holding `-log p(y | x)` under `p_ref`, `p_0` and `p_T`. The core is
numpy-only and framework-agnostic. Section 11 shows, optionally, how to produce those vectors from a
torch model. Nothing in the core imports torch.

**Order of the notebook.** Sections 2 and 3 are the two checks that have to pass before any number
is computed: how the panel was used, and what counts as one audit unit. Sections 4 to 8 are the
computation, the report object, panel sizing and the reconciliation check. Section 9 reproduces the
ARC-TGI run from the paper and asserts the published values, and Section 10 shows the guardrail
refusing the other panel of that same run.


In [1]:
import hashlib
import json
import math
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np

UNIT = "nats"

print("python :", sys.version.split()[0])
print("numpy  :", np.__version__)
print("unit   :", UNIT)

python : 3.9.6
numpy  : 2.0.2
unit   : nats


## 1. Losses in, numbers out

The core takes three vectors of per-audit-unit losses over the *same* panel, in the same order:

- `loss_ref[i] = -log p_ref(y_i | x_i)` under the benchmark-owned reference quote,
- `loss_0[i]   = -log p_0(y_i | x_i)`   under the initial state,
- `loss_T[i]   = -log p_T(y_i | x_i)`   under the terminal state.

A finite-sample bound needs a bounded loss. The construction in the paper forms the full-support
quote `p_alpha = (1 - alpha) p + alpha u` with `u_min > 0`, so `0 <= -log p_alpha <= R` with
`R = -log(alpha * u_min)`. Declare that `R` and the checks below will hold the arrays to it. An
unfloored quote can produce an infinite loss on a single unit, which no concentration argument
covers.


In [2]:
def check_loss_vector(x: Sequence[float], name: str, R: Optional[float] = None) -> np.ndarray:
    """Validate one per-audit-unit loss vector and return it as float64.

    Rejects the three mistakes that silently corrupt the report: log-likelihoods passed with the
    wrong sign, an unfloored quote that produced an infinite loss, and losses that exceed the
    declared range R used by the slack.
    """
    a = np.asarray(x, dtype=np.float64)
    if a.ndim != 1:
        raise ValueError(f"{name}: expected a 1-D vector of per-audit-unit losses, got shape {a.shape}")
    if a.size == 0:
        raise ValueError(f"{name}: empty loss vector")
    if not np.all(np.isfinite(a)):
        bad = int(np.sum(~np.isfinite(a)))
        raise ValueError(
            f"{name}: {bad} non-finite entries. -log p is infinite where the quote assigns zero "
            f"probability to the observed outcome; floor the quote before scoring."
        )
    if np.any(a < 0.0):
        raise ValueError(
            f"{name}: negative entries. These vectors hold -log p(y|x), which is non-negative; "
            f"a sign was dropped."
        )
    if R is not None and np.any(a > R + 1e-9):
        worst = float(a.max())
        raise ValueError(
            f"{name}: max entry {worst:.6f} exceeds the declared loss range R={R:.6f}. Either the "
            f"quote was not floored at the declared alpha, or R was declared too small."
        )
    return a


def aligned_panel(loss_ref, loss_0, loss_T, R: Optional[float] = None) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Validate the three vectors and check they describe one panel, unit for unit."""
    a = check_loss_vector(loss_ref, "loss_ref", R)
    b = check_loss_vector(loss_0, "loss_0", R)
    c = check_loss_vector(loss_T, "loss_T", R)
    if not (a.size == b.size == c.size):
        raise ValueError(
            f"length mismatch: loss_ref={a.size}, loss_0={b.size}, loss_T={c.size}. The three "
            f"vectors must score the same panel, in the same order."
        )
    return a, b, c


def mean_log_loss(x, name: str = "loss", R: Optional[float] = None) -> float:
    """L_Q(p) estimated on the panel: the mean per-audit-unit log loss, in nats."""
    return float(check_loss_vector(x, name, R).mean())


_demo = np.full(8, math.log(4.0))
print("check_loss_vector on a constant quote of 1/4 over 8 units:")
print("  mean log loss = %.6f nats  (= ln 4 = %.6f)" % (mean_log_loss(_demo), math.log(4.0)))
try:
    check_loss_vector([0.3, -0.1, 0.2], "loss_0")
except ValueError as exc:
    print("  rejected as expected:", str(exc).split(".")[0])

check_loss_vector on a constant quote of 1/4 over 8 units:
  mean log loss = 1.386294 nats  (= ln 4 = 1.386294)
  rejected as expected: loss_0: negative entries


## 2. The guardrail: how was the panel used?

Route (c) below prices a **fixed panel**. It is valid only if the panel produced no signal that
influenced training. If the panel's scores steered training in any way, the terminal state was
chosen with knowledge of the panel, the empirical loss at that state is no longer an unbiased read
of `L_Q`, and route (c) does not apply. Route (d) applies instead, with an honest count of the
audit-derived bits that reached training.

This is the most likely error, so it is a check and not a comment. `PanelProvenance` cannot be
built without answering all five channels, and route (c) raises on a panel that steered training.

The five channels, from the paper's accounting: audit scores entering a **reward** or training loss,
deciding **early stopping**, selecting a **checkpoint** to keep or report, selecting a
**hyperparameter**, schedule or architecture, or selecting or reweighting **training data**. The
paper charges scores, thresholds, selected examples, timing, stopping changes and scheduler changes
caused by audit feedback to the transcript.


In [3]:
STEERING_CHANNELS: Tuple[str, ...] = (
    "reward",                     # audit scores entered a training reward or training loss
    "early_stopping",             # audit scores decided when training stopped
    "checkpoint_selection",       # audit scores chose which checkpoint was kept or reported
    "hyperparameter_selection",   # audit scores chose a hyperparameter, schedule or architecture
    "data_selection",             # audit scores chose or reweighted training data
)


class PanelSteeredError(RuntimeError):
    """Raised when a route is requested that the panel's provenance does not support."""


@dataclass(frozen=True)
class PanelProvenance:
    """A declaration of how an audit panel was used relative to training.

    Build this with declare_panel(), which makes every channel a required keyword argument.
    """
    name: str
    influenced_training: Dict[str, bool]
    transcript_bits: Optional[float] = None
    note: str = ""

    def __post_init__(self) -> None:
        missing = [c for c in STEERING_CHANNELS if c not in self.influenced_training]
        if missing:
            raise ValueError(
                "declare every channel explicitly; missing: " + ", ".join(missing)
            )
        unknown = [k for k in self.influenced_training if k not in STEERING_CHANNELS]
        if unknown:
            raise ValueError("unknown channel(s): " + ", ".join(unknown))
        for k, v in self.influenced_training.items():
            if not isinstance(v, bool):
                raise TypeError(
                    f"channel {k!r} must be True or False, got {type(v).__name__}. A truthy "
                    f"placeholder is not a declaration."
                )
        if self.steering_channels and self.transcript_bits is None:
            raise ValueError(
                f"panel {self.name!r} steered training through "
                f"{', '.join(self.steering_channels)}. Declare transcript_bits: an honest count of "
                f"the audit-derived bits exposed to training, so route (d) can price it."
            )
        if self.transcript_bits is not None and self.transcript_bits < 0:
            raise ValueError("transcript_bits must be non-negative")

    @property
    def steering_channels(self) -> Tuple[str, ...]:
        return tuple(c for c in STEERING_CHANNELS if self.influenced_training[c])

    @property
    def sealed(self) -> bool:
        """True when the panel produced no signal that influenced training."""
        return not self.steering_channels

    def describe(self) -> str:
        if self.sealed:
            return f"panel {self.name!r}: sealed, no declared influence on training"
        return (
            f"panel {self.name!r}: steered training through "
            f"{', '.join(self.steering_channels)}; declared transcript {self.transcript_bits:g} bits"
        )


def declare_panel(
    name: str,
    *,
    reward: bool,
    early_stopping: bool,
    checkpoint_selection: bool,
    hyperparameter_selection: bool,
    data_selection: bool,
    transcript_bits: Optional[float] = None,
    note: str = "",
) -> PanelProvenance:
    """Declare how the panel was used. All five channels are required keyword arguments.

    Answer each with True or False for the run being reported. There is no default: a panel whose
    use has not been declared cannot be priced.
    """
    return PanelProvenance(
        name=name,
        influenced_training={
            "reward": reward,
            "early_stopping": early_stopping,
            "checkpoint_selection": checkpoint_selection,
            "hyperparameter_selection": hyperparameter_selection,
            "data_selection": data_selection,
        },
        transcript_bits=transcript_bits,
        note=note,
    )


sealed_example = declare_panel(
    "demo sealed panel",
    reward=False,
    early_stopping=False,
    checkpoint_selection=False,
    hyperparameter_selection=False,
    data_selection=False,
    note="drawn once, scored twice, never read during training",
)
steered_example = declare_panel(
    "demo steering probe",
    reward=True,
    early_stopping=False,
    checkpoint_selection=True,
    hyperparameter_selection=False,
    data_selection=False,
    transcript_bits=4096.0,
)
print(sealed_example.describe())
print(steered_example.describe())

try:
    declare_panel(
        "undeclared",
        reward=True,
        early_stopping=False,
        checkpoint_selection=False,
        hyperparameter_selection=False,
        data_selection=False,
    )
except ValueError as exc:
    print("\nrefused, as expected:\n ", str(exc))

panel 'demo sealed panel': sealed, no declared influence on training
panel 'demo steering probe': steered training through reward, checkpoint_selection; declared transcript 4096 bits

refused, as expected:
  panel 'undeclared' steered training through reward. Declare transcript_bits: an honest count of the audit-derived bits exposed to training, so route (d) can price it.


## 3. What counts as one audit unit

`n` is the number of **independent** audit units. Cells within one grid, tokens within one document,
frames within one episode and turns within one dialogue are dependent, so the conservative unit is
the grid, the document, the episode, the dialogue. Substituting the token or cell count for `n`
shrinks the slack by roughly the square root of the number of sub-units per record and produces a
lower bound the panel does not support.

The archived ARC-TGI audit metric, for instance, averages probability-floored cross-entropy over the
content cells inside each panel item. The per-item value is the conservative unit; the cell count is
not `n`.


In [4]:
DEPENDENT_UNIT_KINDS = {
    "token", "subword", "wordpiece", "character", "char", "byte",
    "cell", "grid_cell", "pixel", "voxel", "patch",
    "frame", "timestep", "step", "turn", "utterance", "sentence", "line",
}
INDEPENDENT_UNIT_KINDS = {
    "example", "document", "item", "episode", "trajectory", "question",
    "problem", "task_instance", "grid", "dialogue", "record", "audit_unit",
}


def check_independent_units(
    n_units: int,
    unit_kind: str,
    *,
    n_records: Optional[int] = None,
    strict: bool = False,
) -> List[str]:
    """Warn when the declared n looks like a sub-unit count rather than an independent-unit count.

    n_records, when given, is the number of independent records (examples, documents, episodes)
    the panel was drawn from. Returns the list of warnings and prints them; strict=True raises
    instead.
    """
    warnings_out: List[str] = []
    kind = str(unit_kind).strip().lower().replace(" ", "_")
    if int(n_units) < 1:
        raise ValueError("n_units must be a positive integer")

    if kind in DEPENDENT_UNIT_KINDS:
        warnings_out.append(
            f"declared unit {kind!r} is a sub-unit of an example, document or episode. Sub-units "
            f"within one record are dependent, so n={n_units} is anti-conservative. Use the number "
            f"of independent records instead."
        )
    elif kind not in INDEPENDENT_UNIT_KINDS:
        warnings_out.append(
            f"unrecognised unit kind {kind!r}. Confirm it is the independent unit: if several of "
            f"them come from one example, document or episode, they are dependent."
        )

    if n_records is not None:
        if int(n_records) < 1:
            raise ValueError("n_records must be a positive integer")
        if n_units > n_records:
            ratio = n_units / float(n_records)
            factor = math.sqrt(ratio)
            warnings_out.append(
                f"n_units={n_units} exceeds n_records={n_records} at {ratio:.1f} sub-units per "
                f"record. The conservative n is {n_records}; using {n_units} understates the slack "
                f"by about a factor of {factor:.1f}."
            )

    if kind in INDEPENDENT_UNIT_KINDS and n_units >= 1_000_000:
        warnings_out.append(
            f"n={n_units} independent {kind}s is a large audit panel. Confirm these are records "
            f"and not tokens, cells or frames."
        )

    if warnings_out:
        if strict:
            raise ValueError("independent-unit check failed:\n  - " + "\n  - ".join(warnings_out))
        for w in warnings_out:
            print("WARNING (independent unit): " + w)
    else:
        print(f"independent-unit check passed: n={n_units} {kind}s")
    return warnings_out


print("-- correct declaration for the ARC-TGI sealed panel:")
check_independent_units(12288, "audit_unit", n_records=12288)
print()
print("-- the same panel declared by its cell count:")
check_independent_units(12288 * 900, "cell", n_records=12288)

-- correct declaration for the ARC-TGI sealed panel:
independent-unit check passed: n=12288 audit_units

-- the same panel declared by its cell count:
WARNING (independent unit): declared unit 'cell' is a sub-unit of an example, document or episode. Sub-units within one record are dependent, so n=11059200 is anti-conservative. Use the number of independent records instead.
WARNING (independent unit): n_units=11059200 exceeds n_records=12288 at 900.0 sub-units per record. The conservative n is 12288; using 11059200 understates the slack by about a factor of 30.0.


## 4. The four slack routes

The slack is the additive error budget in the endpoint inequality. The user declares which route
applies; the notebook computes only the matching formula.

**(a) Complete verifier, `eta`-complete.**

```
Gamma = 2 * eta
```

**(b) Fresh endpoints.** Independent batches of size `m0` and `mT` at the two endpoints, with loss
widths `W0` and `WT`, and `delta0 + deltaT <= delta`:

```
Gamma = W0 * sqrt(ln(2/delta0) / (2*m0)) + WT * sqrt(ln(2/deltaT) / (2*mT))
```

**(c) Fixed panel.** `n` units, loss range `R`, a declared finite candidate set of size `N`:

```
Delta = R * sqrt(ln(2*N/delta) / (2*n))
Gamma = 2 * Delta
```

The two-sided form covers an upper deviation at `p_0` and a lower deviation at `p_T`. Valid only for
a panel that produced no signal that influenced training.

**(d) Reused panel** whose audit-derived transcript takes at most `2^B` values:

```
Gamma = R * sqrt(2 * (B*ln(2) + ln(1/delta)) / n)
```

Route (d) is where a steered panel goes. `B` is a declared budget on the released transcript, not an
estimate of how much information the training run actually used.


In [5]:
@dataclass(frozen=True)
class Slack:
    """A slack value together with the route and parameters that produced it."""
    gamma: float
    route: str
    route_label: str
    params: Dict[str, Any]
    transcript_bits: Optional[float] = None

    def __str__(self) -> str:
        return f"Gamma = {self.gamma:.6f} nats  [route {self.route}: {self.route_label}]"


def slack_complete_verifier(*, eta: float) -> Slack:
    """Route (a): a complete verifier, eta-complete. Gamma = 2*eta."""
    if eta < 0:
        raise ValueError("eta must be non-negative")
    return Slack(2.0 * eta, "a", "complete verifier, eta-complete", {"eta": float(eta)})


def slack_fresh_endpoints(*, m0: int, mT: int, W0: float, WT: float,
                          delta0: float, deltaT: float, delta: float) -> Slack:
    """Route (b): independent fresh batches at the two endpoints."""
    for nm, v in (("m0", m0), ("mT", mT)):
        if int(v) < 1:
            raise ValueError(f"{nm} must be a positive integer")
    for nm, v in (("W0", W0), ("WT", WT)):
        if v <= 0:
            raise ValueError(f"{nm} must be positive")
    for nm, v in (("delta0", delta0), ("deltaT", deltaT), ("delta", delta)):
        if not 0.0 < v < 1.0:
            raise ValueError(f"{nm} must lie in (0, 1)")
    if delta0 + deltaT > delta + 1e-15:
        raise ValueError(
            f"delta0 + deltaT = {delta0 + deltaT:g} exceeds delta = {delta:g}; the two endpoint "
            f"confidences must fit inside the declared budget."
        )
    term0 = W0 * math.sqrt(math.log(2.0 / delta0) / (2.0 * m0))
    termT = WT * math.sqrt(math.log(2.0 / deltaT) / (2.0 * mT))
    return Slack(
        term0 + termT, "b", "fresh endpoints",
        {"m0": int(m0), "mT": int(mT), "W0": float(W0), "WT": float(WT),
         "delta0": float(delta0), "deltaT": float(deltaT), "delta": float(delta),
         "term_initial": term0, "term_terminal": termT},
    )


def panel_radius(*, n: int, R: float, delta: float, N: float) -> float:
    """Route (c) per-endpoint radius: Delta = R*sqrt(ln(2N/delta)/(2n))."""
    if int(n) < 1:
        raise ValueError("n must be a positive integer")
    if R <= 0:
        raise ValueError("R must be positive")
    if not 0.0 < delta < 1.0:
        raise ValueError("delta must lie in (0, 1)")
    if N < 1:
        raise ValueError("N must be at least 1")
    return R * math.sqrt(math.log(2.0 * N / delta) / (2.0 * n))


ROUTE_C_REFUSAL = """Route (c) is not available for panel {name!r}.

The panel produced signal that influenced training through: {channels}.
A panel whose scores steered training is not a fixed-model measurement: the terminal state was
chosen with knowledge of the panel, so its empirical loss there is not an unbiased read of L_Q and
the uniform-deviation event that route (c) prices does not cover it.

Use route (d) with an honest bit count of the audit-derived transcript exposed to training:

    compute_slack("d", provenance=prov, n={n}, R={R!r}, delta={delta!r}, transcript_bits=B)

Count every score, threshold, selected example, timing signal, stopping change and scheduler change
caused by audit feedback. If the released bits cannot be counted, the panel does not support a lower
bound at all."""


def slack_fixed_panel(*, n: int, R: float, delta: float, N: float,
                      provenance: PanelProvenance) -> Slack:
    """Route (c): fixed panel, Gamma = 2*Delta. Refuses a panel that steered training."""
    if not isinstance(provenance, PanelProvenance):
        raise PanelSteeredError(
            "route (c) requires a declared PanelProvenance. Build one with declare_panel() and "
            "answer all five steering channels."
        )
    if not provenance.sealed:
        raise PanelSteeredError(ROUTE_C_REFUSAL.format(
            name=provenance.name, channels=", ".join(provenance.steering_channels),
            n=n, R=R, delta=delta,
        ))
    Delta = panel_radius(n=n, R=R, delta=delta, N=N)
    return Slack(
        2.0 * Delta, "c", "fixed panel, declared candidate set",
        {"n": int(n), "R": float(R), "delta": float(delta), "N": float(N), "Delta": Delta,
         "panel": provenance.name},
    )


def slack_reused_panel(*, n: int, R: float, delta: float, transcript_bits: float) -> Slack:
    """Route (d): reused panel with a transcript taking at most 2^B values."""
    if int(n) < 1:
        raise ValueError("n must be a positive integer")
    if R <= 0:
        raise ValueError("R must be positive")
    if not 0.0 < delta < 1.0:
        raise ValueError("delta must lie in (0, 1)")
    if transcript_bits is None or transcript_bits < 0:
        raise ValueError("transcript_bits must be a declared non-negative bit budget")
    B = float(transcript_bits)
    gamma = R * math.sqrt(2.0 * (B * math.log(2.0) + math.log(1.0 / delta)) / n)
    return Slack(
        gamma, "d", "reused panel, bounded audit transcript",
        {"n": int(n), "R": float(R), "delta": float(delta), "transcript_bits": B},
        transcript_bits=B,
    )


_ROUTES = {
    "a": slack_complete_verifier,
    "b": slack_fresh_endpoints,
    "c": slack_fixed_panel,
    "d": slack_reused_panel,
}


def compute_slack(route: str, *, provenance: Optional[PanelProvenance] = None, **params) -> Slack:
    """Declare the route and its parameters; returns the matching slack.

    provenance is required for routes (c) and (d). Route (c) refuses a panel that steered training.
    """
    key = str(route).strip().lower().lstrip("(").rstrip(")")
    if key not in _ROUTES:
        raise ValueError(f"unknown route {route!r}; declare one of 'a', 'b', 'c', 'd'")
    if key == "c":
        return slack_fixed_panel(provenance=provenance, **params)
    if key == "d":
        if "transcript_bits" not in params:
            if provenance is None or provenance.transcript_bits is None:
                raise ValueError(
                    "route (d) needs transcript_bits, either as an argument or on the provenance"
                )
            params["transcript_bits"] = provenance.transcript_bits
        return slack_reused_panel(**params)
    return _ROUTES[key](**params)


print(compute_slack("a", eta=0.02))
print(compute_slack("b", m0=4096, mT=4096, W0=4.605170186, WT=4.605170186,
                    delta0=0.025, deltaT=0.025, delta=0.05))
print(compute_slack("c", provenance=sealed_example, n=12288, R=4.605170186, delta=0.05, N=1))
print(compute_slack("d", provenance=steered_example, n=12288, R=4.605170186, delta=0.05))

Gamma = 0.040000 nats  [route a: complete verifier, eta-complete]
Gamma = 0.213019 nats  [route b: fresh endpoints]
Gamma = 0.112841 nats  [route c: fixed panel, declared candidate set]
Gamma = 3.132144 nats  [route d: reused panel, bounded audit transcript]


## 5. The five numbers

`five_numbers()` takes the three loss vectors and a `Slack`, and returns the report quantities. It
is the whole computation: everything else in this notebook is a check on its inputs or a way of
publishing its outputs.


In [6]:
@dataclass(frozen=True)
class FiveNumbers:
    """The five reported numbers, all in nats except the dimensionless ratio."""
    terminal_level: float          # B = L_Q(p_ref) - L_Q(p_T)
    signed_gain: float             # G = L_Q(p_0) - L_Q(p_T)
    slack: float                   # Gamma
    lower_bound: float             # G - Gamma
    discrimination_ratio: float    # G / Gamma
    level_ref: float
    level_initial: float
    level_terminal: float
    n_units: int
    route: str

    def summary(self) -> str:
        lines = [
            f"  1. terminal level        B = {self.terminal_level:+.4f} nats",
            f"  2. signed gain           G = {self.signed_gain:+.4f} nats",
            f"  3. slack             Gamma = {self.slack:.6f} nats   [route {self.route}]",
            f"  4. lower bound   G - Gamma = {self.lower_bound:+.4f} nats",
            f"  5. ratio         G / Gamma = {self.discrimination_ratio:.2f}",
        ]
        if self.lower_bound > 0.0:
            lines.append(
                f"     -> improved by at least {self.lower_bound:.4f} nats on the declared target"
            )
        else:
            lines.append(
                "     -> no improvement is demonstrated at this resolution"
            )
        return "\n".join(lines)


def five_numbers(loss_ref, loss_0, loss_T, slack: Slack, *, R: Optional[float] = None) -> FiveNumbers:
    """Compute the five reported numbers from three per-audit-unit loss vectors and a slack."""
    if not isinstance(slack, Slack):
        raise TypeError("slack must be a Slack; build one with compute_slack()")
    a, b, c = aligned_panel(loss_ref, loss_0, loss_T, R)
    L_ref, L_0, L_T = float(a.mean()), float(b.mean()), float(c.mean())
    B = L_ref - L_T
    G = L_0 - L_T
    gamma = float(slack.gamma)
    lower = G - gamma
    if gamma > 0.0:
        ratio = G / gamma
    else:
        ratio = math.inf if G > 0.0 else math.nan
    return FiveNumbers(
        terminal_level=B, signed_gain=G, slack=gamma, lower_bound=lower,
        discrimination_ratio=ratio, level_ref=L_ref, level_initial=L_0, level_terminal=L_T,
        n_units=int(a.size), route=slack.route,
    )


_n = 4096
_rng = np.random.default_rng(0)
_ref = np.full(_n, math.log(10.0))
_l0 = np.clip(_rng.normal(2.10, 0.30, _n), 0.0, 4.605170186)
_lT = np.clip(_rng.normal(1.60, 0.30, _n), 0.0, 4.605170186)
_prov = declare_panel("toy sealed panel", reward=False, early_stopping=False,
                      checkpoint_selection=False, hyperparameter_selection=False,
                      data_selection=False)
_slack = compute_slack("c", provenance=_prov, n=_n, R=4.605170186, delta=0.05, N=1)
print("toy panel, n = %d" % _n)
print(five_numbers(_ref, _l0, _lT, _slack, R=4.605170186).summary())

toy panel, n = 4096
  1. terminal level        B = +0.6966 nats
  2. signed gain           G = +0.4892 nats
  3. slack             Gamma = 0.195447 nats   [route c]
  4. lower bound   G - Gamma = +0.2938 nats
  5. ratio         G / Gamma = 2.50
     -> improved by at least 0.2938 nats on the declared target


## 6. The report object

The paper's Discussion fixes what a public report contains: the terminal level against the benchmark
reference, the cumulative training gain, the target and masking-policy versions, the primitive audit
unit, the confidence or completeness slack, the cumulative audit-derived transcript bits, and the
commitment record. `audit_cp_report()` emits exactly those fields as JSON, plus the two derived
numbers and a digest that binds the report to the loss vectors it was computed from.

The commitment record is the protocol's second step: the state is committed before the randomness
that determines its challenge is drawn. Supply the record you actually hold.


In [7]:
def evidence_digest(loss_ref, loss_0, loss_T) -> str:
    """sha256 over the three loss vectors, so a report is bound to the evidence it came from."""
    a, b, c = aligned_panel(loss_ref, loss_0, loss_T)
    h = hashlib.sha256()
    for name, arr in (("ref", a), ("p0", b), ("pT", c)):
        h.update(name.encode("utf-8"))
        h.update(np.ascontiguousarray(arr, dtype="<f8").tobytes())
    return "sha256:" + h.hexdigest()


REQUIRED_COMMITMENT_KEYS = ("state_commitment", "committed_at", "challenge_drawn_after_commitment")


def _finite_or_none(x: float) -> Optional[float]:
    return float(x) if math.isfinite(x) else None


def audit_cp_report(
    loss_ref, loss_0, loss_T, slack: Slack,
    *,
    target_version: str,
    reference_version: str,
    masking_policy_version: str,
    audit_unit: str,
    commitment_record: Dict[str, Any],
    R: Optional[float] = None,
    n_independent_units: Optional[int] = None,
    component_notes: Optional[List[str]] = None,
) -> Dict[str, Any]:
    """Compute the five numbers and emit the public report object."""
    missing = [k for k in REQUIRED_COMMITMENT_KEYS if k not in commitment_record]
    if missing:
        raise ValueError(
            "commitment_record is missing " + ", ".join(missing) +
            ". The report is the scalar plus the evidence process that gives it meaning."
        )
    if commitment_record["challenge_drawn_after_commitment"] is not True:
        print(
            "WARNING (commitment): challenge randomness was not drawn after the state was "
            "committed. The endpoint bound assumes the audited state was fixed before its "
            "challenge was drawn."
        )
    fn = five_numbers(loss_ref, loss_0, loss_T, slack, R=R)
    n_units = int(n_independent_units if n_independent_units is not None else fn.n_units)

    return {
        "unit": UNIT,
        "terminal_level": round(fn.terminal_level, 6),
        "signed_gain": round(fn.signed_gain, 6),
        "lower_bound_on_improvement": round(fn.lower_bound, 6),
        "discrimination_ratio": (None if not math.isfinite(fn.discrimination_ratio)
                                 else round(fn.discrimination_ratio, 4)),
        "target_version": target_version,
        "reference_version": reference_version,
        "masking_policy_version": masking_policy_version,
        "audit_unit": audit_unit,
        "n_independent_units": n_units,
        "slack": round(fn.slack, 6),
        "slack_route": slack.route,
        "slack_route_label": slack.route_label,
        "slack_parameters": {k: (_finite_or_none(v) if isinstance(v, float) else v)
                             for k, v in slack.params.items()},
        "transcript_bits": (None if slack.transcript_bits is None else float(slack.transcript_bits)),
        "commitment_record": dict(commitment_record),
        "levels": {
            "L_reference": round(fn.level_ref, 6),
            "L_initial": round(fn.level_initial, 6),
            "L_terminal": round(fn.level_terminal, 6),
        },
        "evidence_digest": evidence_digest(loss_ref, loss_0, loss_T),
        "notes": list(component_notes or []),
    }


_report_demo = audit_cp_report(
    _ref, _l0, _lT, _slack,
    target_version="toy-target/v1",
    reference_version="uniform-over-10/v1",
    masking_policy_version="full-transcript/v1",
    audit_unit="one synthetic audit example, mean log loss over the example",
    n_independent_units=_n,
    commitment_record={
        "state_commitment": "sha256:0000000000000000000000000000000000000000000000000000000000000000",
        "committed_at": "1970-01-01T00:00:00Z",
        "challenge_drawn_after_commitment": True,
    },
    R=4.605170186,
)
print(json.dumps(_report_demo, indent=2)[:900] + "\n  ...")

{
  "unit": "nats",
  "terminal_level": 0.69665,
  "signed_gain": 0.489226,
  "lower_bound_on_improvement": 0.293779,
  "discrimination_ratio": 2.5031,
  "target_version": "toy-target/v1",
  "reference_version": "uniform-over-10/v1",
  "masking_policy_version": "full-transcript/v1",
  "audit_unit": "one synthetic audit example, mean log loss over the example",
  "n_independent_units": 4096,
  "slack": 0.195447,
  "slack_route": "c",
  "slack_route_label": "fixed panel, declared candidate set",
  "slack_parameters": {
    "n": 4096,
    "R": 4.605170186,
    "delta": 0.05,
    "N": 1.0,
    "Delta": 0.09772325952890798,
    "panel": "toy sealed panel"
  },
  "transcript_bits": null,
  "commitment_record": {
    "state_commitment": "sha256:0000000000000000000000000000000000000000000000000000000000000000",
    "committed_at": "1970-01-01T00:00:00Z",
    "challenge_drawn_after_commitment": t
  ...


## 7. Resolution before the run

The route (c) slack is fixed by the panel size, the loss range, the confidence level and the
declared candidate count, and by nothing about the model or the run. It is therefore available
before training begins, and a panel can be sized to the improvement it has to resolve.

`resolution_before_run(n, R, delta, N)` returns the slack. `panel_size_for_resolution()` inverts it.


In [8]:
def resolution_before_run(*, n: int, R: float, delta: float, N: float) -> Dict[str, float]:
    """Slack available from a panel of n units, before any training. Route (c) arithmetic."""
    Delta = panel_radius(n=n, R=R, delta=delta, N=N)
    return {"n": int(n), "R": float(R), "delta": float(delta), "N": float(N),
            "radius": Delta, "slack": 2.0 * Delta}


def panel_size_for_resolution(*, gamma_target: float, R: float, delta: float, N: float) -> int:
    """Smallest n whose route (c) slack is at most gamma_target."""
    if gamma_target <= 0:
        raise ValueError("gamma_target must be positive")
    Delta = gamma_target / 2.0
    n = R * R * math.log(2.0 * N / delta) / (2.0 * Delta * Delta)
    return int(math.ceil(n))


R_ARC, DELTA_ARC = 4.605170186, 0.05

print("Sizing a panel at R = %.6f (probability floor 0.01), delta = %.2f, N = 1." % (R_ARC, DELTA_ARC))
print()
print("  smallest improvement to resolve      required n")
for g in (0.50, 0.25, 0.15, 0.112841, 0.10, 0.05, 0.02):
    need = panel_size_for_resolution(gamma_target=g, R=R_ARC, delta=DELTA_ARC, N=1)
    print("  %-36s %10d" % ("%.6f nats" % g, need))

print()
print("  panel size        radius Delta        slack Gamma")
for n in (200, 1024, 4096, 12288, 65536, 262144):
    r = resolution_before_run(n=n, R=R_ARC, delta=DELTA_ARC, N=1)
    print("  %-16d %14.6f %18.6f" % (n, r["radius"], r["slack"]))

print()
print("  Declared candidate set N at n = 12288:")
for label, N in (("1", 1.0), ("10^6", 1e6), ("10^9", 1e9), ("2^1000", 2.0 ** 1000)):
    r = resolution_before_run(n=12288, R=R_ARC, delta=DELTA_ARC, N=N)
    print("  N = %-10s radius %.6f   slack %.6f" % (label, r["radius"], r["slack"]))

_need = panel_size_for_resolution(gamma_target=0.10, R=R_ARC, delta=DELTA_ARC, N=1)
print()
print("Worked sizing example. A benchmark that has to resolve 0.10 nats at R = %.6f, delta = %.2f" % (R_ARC, DELTA_ARC))
print("and N = 1 needs n = %d independent audit units. The 12,288-unit ARC-TGI panel resolves" % _need)
print("%.6f nats, so it separates a 0.74-nat gain and would not separate a 0.10-nat one." %
      resolution_before_run(n=12288, R=R_ARC, delta=DELTA_ARC, N=1)["slack"])

Sizing a panel at R = 4.605170 (probability floor 0.01), delta = 0.05, N = 1.

  smallest improvement to resolve      required n
  0.500000 nats                               626
  0.250000 nats                              2504
  0.150000 nats                              6954
  0.112841 nats                             12289
  0.100000 nats                             15647
  0.050000 nats                             62586
  0.020000 nats                            391162

  panel size        radius Delta        slack Gamma
  200                    0.442245           0.884490
  1024                   0.195447           0.390893
  4096                   0.097723           0.195447
  12288                  0.056421           0.112841
  65536                  0.024431           0.048862
  262144                 0.012215           0.024431

  Declared candidate set N at n = 12288:
  N = 1          radius 0.056421   slack 0.112841
  N = 10^6       radius 0.122903   slack 0.245807
  N = 10

## 8. Reconciliation: increments against endpoints

Audit compression progress is a potential difference, `r_t = B_Q(p_t) - B_Q(p_{t-1})`, so the
increments telescope and a reported total is checked against the two endpoint levels alone. That is
what lets a third party check a total without seeing the trajectory: verification does not depend on
the number of steps, and the intermediate states never have to be disclosed.


In [9]:
def reconcile_increments(increments, *, level_initial: float, level_terminal: float,
                         tol: float = 1e-9) -> Dict[str, Any]:
    """Check that per-step increments sum to the endpoint difference B_Q(p_T) - B_Q(p_0)."""
    r = np.asarray(increments, dtype=np.float64)
    if r.ndim != 1:
        raise ValueError("increments must be a 1-D vector of per-step values")
    if not np.all(np.isfinite(r)):
        raise ValueError("increments contain non-finite entries")
    total = float(r.sum())
    endpoint = float(level_terminal) - float(level_initial)
    residual = total - endpoint
    ok = abs(residual) <= tol * max(1.0, abs(endpoint))
    out = {"steps": int(r.size), "sum_of_increments": total, "endpoint_difference": endpoint,
           "residual": residual, "reconciles": bool(ok),
           "negative_increments": int(np.sum(r < 0.0))}
    status = "reconciles" if ok else "DOES NOT RECONCILE"
    print("%s: %d steps, sum %+.9f against endpoint difference %+.9f, residual %+.2e"
          % (status, out["steps"], total, endpoint, residual))
    return out


_rng2 = np.random.default_rng(7)
_G_true = 0.73658
_steps = _rng2.normal(0.0, 0.05, 400)
_steps += (_G_true - _steps.sum()) / _steps.size
_L_ref, _L0v, _LTv = math.log(11.0), 2.48413, 1.74755
_B0, _BT = _L_ref - _L0v, _L_ref - _LTv

print("A 400-step trajectory, %d of whose increments are negative:" % int(np.sum(_steps < 0)))
_rec = reconcile_increments(_steps, level_initial=_B0, level_terminal=_BT)
print()
print("Same ledger with one increment overstated by 0.05 nats:")
_tampered = _steps.copy()
_tampered[123] += 0.05
_bad = reconcile_increments(_tampered, level_initial=_B0, level_terminal=_BT)
print()
print("A third party checks the total against the two endpoint levels alone, so a trajectory can be")
print("verified without being disclosed and the check does not grow with its length.")
assert _rec["reconciles"] and not _bad["reconciles"]

A 400-step trajectory, 189 of whose increments are negative:
reconciles: 400 steps, sum +0.736580000 against endpoint difference +0.736580000, residual +2.22e-16

Same ledger with one increment overstated by 0.05 nats:
DOES NOT RECONCILE: 400 steps, sum +0.786580000 against endpoint difference +0.736580000, residual +5.00e-02

A third party checks the total against the two endpoint levels alone, so a trajectory can be
verified without being disclosed and the check does not grow with its length.


## 9. Worked example: the ARC-TGI run from the paper

The paper's worked example is the ARC-TGI curriculum run, scored on the sealed panel that training
never touched. Its declared parameters are

```
n = 12288 audit units      R = 4.605170186 = -ln 0.01      delta = 0.05      N = 1      route (c)
```

and the published values are `Delta = 0.056421`, `Gamma = 0.112841`, mean `L_0 = 2.4841`, mean
`L_T = 1.7476`, mean `G = 0.7366`, lower bound `0.6237`, discrimination ratio `6.53`, with per-seed
lower bounds from `0.5266` to `0.6981`, positive in 20 of 20 seeds.

The per-seed endpoint levels come from the record shipped with this repository,
`experiments/results_data/c1_panelB_initial_loss.json`, regenerated by
`experiments/c1_panelB_initial_loss.py`. For each seed it holds the initial and terminal audit level
of panel B: the sealed 12,288-unit panel, scored once after the final training step, disjoint from
the probe panel that the per-step scheduler reward read.

Two points about what is and is not pinned down. The record ships per-seed *levels*, and the five
numbers depend on the loss vectors only through their means, so per-unit vectors carrying those
levels reproduce the report exactly; the per-unit dispersion constructed below enters nothing. And
the published values fix the gain, the slack, the lower bound and the ratio, while the terminal level
`B` also needs a declared benchmark reference: we declare the uniform quote over the 11 ARC-TGI
colour classes, `L_Q(p_ref) = ln 11 = 2.397895` nats, and report `B` against it.


In [10]:
VALIDATION_TARGET = {
    "n": 12288, "R": 4.605170186, "delta": 0.05, "N": 1, "route": "c",
    "Delta": 0.056421, "Gamma": 0.112841,
    "mean_L0": 2.4841, "mean_LT": 1.7476, "mean_G": 0.7366,
    "lower_bound": 0.6237, "discrimination_ratio": 6.53,
    "per_seed_lower_bound_min": 0.5266, "per_seed_lower_bound_max": 0.6981,
    "seeds_positive": 20, "seeds": 20,
}


def _find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "experiments").is_dir() and (p / "README.md").is_file():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd().resolve())

SEED_FILE_DIRS = [
    REPO_ROOT / "experiments" / "results_data",
    REPO_ROOT / "experiments" / "data",
    REPO_ROOT / "experiments",
    REPO_ROOT / "data",
    REPO_ROOT / "results",
    REPO_ROOT / "notebooks" / "data",
]
_L0_ALIASES = ("l0", "l_0", "loss_0", "loss0", "initial_loss", "l_initial", "mean_l0", "l0_mean",
               "audit_loss_initial", "floored_ce_initial", "ce_initial", "ce_0", "initial")
_LT_ALIASES = ("lt", "l_t", "loss_t", "losst", "terminal_loss", "l_terminal", "mean_lt", "lt_mean",
               "audit_loss_terminal", "floored_ce_terminal", "ce_terminal", "ce_t", "terminal",
               "final_audit_loss")
_SEED_ALIASES = ("seed", "seed_id", "run", "run_id", "replicate")


def _pick(keys, aliases):
    lowered = {str(k).strip().lower(): k for k in keys}
    for a in aliases:
        if a in lowered:
            return lowered[a]
    return None


def _rows_to_arrays(rows):
    """Turn a list of per-seed dicts into (seeds, L0, LT), dropping rows not marked ok."""
    rows = [r for r in rows if isinstance(r, dict)]
    rows = [r for r in rows if str(r.get("status", "ok")).lower() == "ok"]
    if len(rows) < 2:
        return None
    k0 = _pick(rows[0].keys(), _L0_ALIASES)
    kT = _pick(rows[0].keys(), _LT_ALIASES)
    if k0 is None or kT is None:
        return None
    ks = _pick(rows[0].keys(), _SEED_ALIASES)
    try:
        seeds = [int(float(r[ks])) for r in rows] if ks else list(range(len(rows)))
        L0 = [float(r[k0]) for r in rows]
        LT = [float(r[kT]) for r in rows]
    except (KeyError, TypeError, ValueError):
        return None
    order = np.argsort(np.array(seeds))
    return (np.array(seeds, dtype=int)[order],
            np.array(L0, dtype=np.float64)[order],
            np.array(LT, dtype=np.float64)[order])


def _row_blocks(obj):
    """Yield the plausible per-seed row lists inside a parsed JSON document."""
    if isinstance(obj, list):
        yield obj
        return
    if not isinstance(obj, dict):
        return
    for key in ("per_seed", "seeds", "rows", "results", "per_seed_endpoints", "data"):
        block = obj.get(key)
        if isinstance(block, list):
            yield block
        elif isinstance(block, dict):
            yield [dict(r, seed=r.get("seed", k))
                   for k, r in block.items() if isinstance(r, dict)]


def _load_candidate(path: Path):
    try:
        if path.suffix.lower() == ".csv":
            import csv
            with path.open(newline="") as fh:
                return _rows_to_arrays(list(csv.DictReader(fh)))
        if path.suffix.lower() == ".json":
            obj = json.loads(path.read_text())
            for block in _row_blocks(obj):
                got = _rows_to_arrays(block)
                if got is not None:
                    return got
            if isinstance(obj, dict):
                k0, kT = _pick(obj.keys(), _L0_ALIASES), _pick(obj.keys(), _LT_ALIASES)
                if k0 and kT and isinstance(obj[k0], list):
                    ks = _pick(obj.keys(), _SEED_ALIASES)
                    seeds = (np.array([int(s) for s in obj[ks]], dtype=int) if ks
                             else np.arange(len(obj[k0])))
                    return (seeds, np.asarray(obj[k0], dtype=np.float64),
                            np.asarray(obj[kT], dtype=np.float64))
    except Exception as exc:
        print("  skipped %s (%s)" % (path.name, exc))
    return None


def find_seed_endpoint_file():
    """Find the shipped per-seed endpoint record: per-seed initial and terminal audit levels."""
    for d in SEED_FILE_DIRS:
        if not d.is_dir():
            continue
        for path in sorted(d.glob("*")):
            if path.suffix.lower() not in (".csv", ".json"):
                continue
            got = _load_candidate(path)
            if got is not None and got[1].size >= 2:
                meta = {}
                if path.suffix.lower() == ".json":
                    obj = json.loads(path.read_text())
                    if isinstance(obj, dict):
                        meta = obj
                return got + (str(path.relative_to(REPO_ROOT)), meta)
    return None


def load_arc_tgi_endpoints():
    """Return (seeds, L0, LT, source, metadata) from the shipped per-seed endpoint record."""
    print("repository root:", REPO_ROOT)
    found = find_seed_endpoint_file()
    if found is None:
        raise FileNotFoundError(
            "no per-seed endpoint record found under:\n    "
            + "\n    ".join(str(d) for d in SEED_FILE_DIRS)
            + "\nThe ARC-TGI record ships as experiments/results_data/c1_panelB_initial_loss.json"
              " and is regenerated by experiments/c1_panelB_initial_loss.py."
        )
    seeds, L0, LT, src, meta = found
    print("per-seed endpoints: %s (%d seeds)" % (src, seeds.size))
    for key in ("measurement", "generated_by"):
        if key in meta:
            print("  %-13s %s" % (key + ":", meta[key]))
    panel = meta.get("panel", {}) if isinstance(meta, dict) else {}
    if panel:
        print("  panel:        %s (sealed=%s, scored_during_training=%s)"
              % (panel.get("name"), panel.get("sealed"), panel.get("scored_during_training")))
    loss = meta.get("loss", {}) if isinstance(meta, dict) else {}
    if loss:
        print("  loss:         %s" % loss.get("name"))
        print("  range R:      %.9f" % float(loss.get("range_R", float("nan"))))
    return seeds, L0, LT, src, meta


ARC_SEEDS, ARC_L0, ARC_LT, ARC_SOURCE, ARC_META = load_arc_tgi_endpoints()
print()
print("seeds: %d   mean L_0 = %.6f   mean L_T = %.6f"
      % (ARC_SEEDS.size, ARC_L0.mean(), ARC_LT.mean()))

repository root: /Users/polaris/Documents/Epistemology and Zetesis/Noumenal/verifiable_RSI/audit-compression-progress
per-seed endpoints: experiments/results_data/c1_panelB_initial_loss.json (20 seeds)
  measurement:  L_B(theta_0): audit level of the untrained model on the sealed panel B
  generated_by: experiments/c1_panelB_initial_loss.py
  panel:        B (sealed=True, scored_during_training=False)
  loss:         floored cross-entropy, content cells only (pad token excluded)
  range R:      4.605170186

seeds: 20   mean L_0 = 2.484117   mean L_T = 1.747562


In [11]:
ARC_N = VALIDATION_TARGET["n"]
ARC_R = VALIDATION_TARGET["R"]
ARC_DELTA = VALIDATION_TARGET["delta"]
ARC_NCAND = VALIDATION_TARGET["N"]
ARC_L_REF = math.log(11.0)          # uniform quote over the 11 ARC-TGI colour classes


def losses_with_mean(target_mean: float, n: int, R: float, spread: float, rng) -> np.ndarray:
    """A per-unit loss vector of length n whose mean is exactly target_mean and whose entries lie
    in [0, R]. Only the mean enters the five numbers; the dispersion is illustrative."""
    g = rng.standard_normal(n)
    g = (g - g.mean()) / g.std()
    v = target_mean + spread * g
    if v.min() < 0.0 or v.max() > R:
        raise ValueError("spread too large for the declared loss range")
    return v


arc_panel = declare_panel(
    "ARC-TGI sealed audit panel",
    reward=False,
    early_stopping=False,
    checkpoint_selection=False,
    hyperparameter_selection=False,
    data_selection=False,
    note=("held out for the whole run; the per-step scheduler reward came from a separate "
          "512-example probe, priced under route (d) in Section 10"),
)
print(arc_panel.describe())
# Cross-check the declaration against the shipped record. The declaration above is the user's, not
# the file's: reading a provenance out of a data file would be the bypass this guardrail exists to
# prevent. What the record can do is contradict the declaration.
_rec_panel = ARC_META.get("panel", {}) if isinstance(ARC_META, dict) else {}
if _rec_panel:
    if _rec_panel.get("sealed") is not True or _rec_panel.get("scored_during_training") is not False:
        raise PanelSteeredError(
            "the shipped record contradicts the declaration above: panel %r has sealed=%r and "
            "scored_during_training=%r, so route (c) does not apply."
            % (_rec_panel.get("name"), _rec_panel.get("sealed"),
               _rec_panel.get("scored_during_training")))
    print("record cross-check: panel %r sealed, not scored during training"
          % _rec_panel.get("name"))

_rec_units = set()
for _blk in _row_blocks(ARC_META):
    for _r in _blk:
        if isinstance(_r, dict) and "n_panel" in _r:
            _rec_units.add(int(_r["n_panel"]))
if _rec_units:
    assert _rec_units == {ARC_N}, \
        "record panel sizes %r disagree with the declared n = %d" % (sorted(_rec_units), ARC_N)
    print("record cross-check: every seed scored n = %d independent audit units" % ARC_N)

_rec_R = float(ARC_META.get("loss", {}).get("range_R", ARC_R)) if isinstance(ARC_META, dict) else ARC_R
assert abs(_rec_R - ARC_R) < 1e-6, \
    "record loss range %.9f disagrees with the declared R = %.9f" % (_rec_R, ARC_R)
print("record cross-check: loss range R = %.9f" % _rec_R)
print()
check_independent_units(ARC_N, "audit_unit", n_records=ARC_N)
print()

arc_slack = compute_slack("c", provenance=arc_panel, n=ARC_N, R=ARC_R, delta=ARC_DELTA, N=ARC_NCAND)
arc_Delta = arc_slack.params["Delta"]
print("route (c) at n = %d, R = %.9f, delta = %.2f, N = %d" % (ARC_N, ARC_R, ARC_DELTA, ARC_NCAND))
print("  radius Delta = %.6f nats" % arc_Delta)
print("  slack  Gamma = %.6f nats" % arc_slack.gamma)

_rng_arc = np.random.default_rng(20260101)
loss_ref_arc = np.full(ARC_N, ARC_L_REF)
per_seed = []
for s, l0, lt in zip(ARC_SEEDS, ARC_L0, ARC_LT):
    v0 = losses_with_mean(float(l0), ARC_N, ARC_R, 0.30, _rng_arc)
    vT = losses_with_mean(float(lt), ARC_N, ARC_R, 0.30, _rng_arc)
    per_seed.append((int(s), five_numbers(loss_ref_arc, v0, vT, arc_slack, R=ARC_R)))

print()
print("  seed      L_0       L_T         G     G - Gamma    G / Gamma")
for s, fn in per_seed:
    print("  %4d  %8.4f  %8.4f  %8.4f    %8.4f     %8.2f"
          % (s, fn.level_initial, fn.level_terminal, fn.signed_gain, fn.lower_bound,
             fn.discrimination_ratio))

per_seed_lb = np.array([fn.lower_bound for _, fn in per_seed])
print()
print("  per-seed lower bound: min %.4f, max %.4f, positive in %d of %d seeds"
      % (per_seed_lb.min(), per_seed_lb.max(), int((per_seed_lb > 0).sum()), per_seed_lb.size))

panel 'ARC-TGI sealed audit panel': sealed, no declared influence on training
record cross-check: panel 'B' sealed, not scored during training
record cross-check: every seed scored n = 12288 independent audit units
record cross-check: loss range R = 4.605170186

independent-unit check passed: n=12288 audit_units

route (c) at n = 12288, R = 4.605170186, delta = 0.05, N = 1
  radius Delta = 0.056421 nats
  slack  Gamma = 0.112841 nats

  seed      L_0       L_T         G     G - Gamma    G / Gamma
     0    2.4619    1.7653    0.6966      0.5838         6.17
     1    2.5175    1.7146    0.8029      0.6900         7.12
     2    2.4847    1.8094    0.6754      0.5625         5.98
     3    2.5108    1.7483    0.7625      0.6497         6.76
     4    2.5544    1.7717    0.7826      0.6698         6.94
     5    2.4959    1.7802    0.7158      0.6029         6.34
     6    2.4339    1.7944    0.6395      0.5266         5.67
     7    2.4416    1.7133    0.7284      0.6155         6.45
  

In [12]:
# Pooled report: the panel scored at the mean initial and mean terminal level over the 20 seeds.
loss_0_arc = losses_with_mean(float(ARC_L0.mean()), ARC_N, ARC_R, 0.30, _rng_arc)
loss_T_arc = losses_with_mean(float(ARC_LT.mean()), ARC_N, ARC_R, 0.30, _rng_arc)
arc_five = five_numbers(loss_ref_arc, loss_0_arc, loss_T_arc, arc_slack, R=ARC_R)

print("ARC-TGI, mean over %d seeds, scored on the sealed panel" % ARC_SEEDS.size)
print("  L_Q(p_ref) = %.6f   L_Q(p_0) = %.6f   L_Q(p_T) = %.6f"
      % (arc_five.level_ref, arc_five.level_initial, arc_five.level_terminal))
print()
print(arc_five.summary())

# ---------------------------------------------------------------------- assertions
V = VALIDATION_TARGET
assert abs(arc_Delta - V["Delta"]) < 5e-7, "radius: %.9f" % arc_Delta
assert abs(arc_slack.gamma - V["Gamma"]) < 5e-7, "slack: %.9f" % arc_slack.gamma
assert abs(arc_five.level_initial - V["mean_L0"]) < 5e-5, "mean L_0: %.6f" % arc_five.level_initial
assert abs(arc_five.level_terminal - V["mean_LT"]) < 5e-5, "mean L_T: %.6f" % arc_five.level_terminal
assert abs(arc_five.signed_gain - V["mean_G"]) < 5e-5, "mean G: %.6f" % arc_five.signed_gain
assert abs(arc_five.lower_bound - V["lower_bound"]) < 5e-5, "lower bound: %.6f" % arc_five.lower_bound
assert abs(arc_five.discrimination_ratio - V["discrimination_ratio"]) < 5e-3, \
    "ratio: %.4f" % arc_five.discrimination_ratio
assert ARC_SEEDS.size == V["seeds"], "seed count: %d" % ARC_SEEDS.size
assert abs(per_seed_lb.min() - V["per_seed_lower_bound_min"]) < 5e-5, "min lb: %.6f" % per_seed_lb.min()
assert abs(per_seed_lb.max() - V["per_seed_lower_bound_max"]) < 5e-5, "max lb: %.6f" % per_seed_lb.max()
assert int((per_seed_lb > 0).sum()) == V["seeds_positive"]

# The reported total is checked against the two endpoint levels alone.
_arc_rec = reconcile_increments(
    np.full(5000, arc_five.signed_gain / 5000.0),
    level_initial=arc_five.level_ref - arc_five.level_initial,
    level_terminal=arc_five.level_ref - arc_five.level_terminal,
)
assert _arc_rec["reconciles"]

print()
print("VALIDATION PASSED against the published ARC-TGI values.")
print("  Delta            %.6f   (target %.6f)" % (arc_Delta, V["Delta"]))
print("  Gamma            %.6f   (target %.6f)" % (arc_slack.gamma, V["Gamma"]))
print("  mean L_0         %.4f     (target %.4f)" % (arc_five.level_initial, V["mean_L0"]))
print("  mean L_T         %.4f     (target %.4f)" % (arc_five.level_terminal, V["mean_LT"]))
print("  mean G           %.4f     (target %.4f)" % (arc_five.signed_gain, V["mean_G"]))
print("  lower bound      %.4f     (target %.4f)" % (arc_five.lower_bound, V["lower_bound"]))
print("  ratio            %.2f       (target %.2f)" % (arc_five.discrimination_ratio, V["discrimination_ratio"]))
print("  per-seed lb      %.4f to %.4f (target %.4f to %.4f), positive %d/%d"
      % (per_seed_lb.min(), per_seed_lb.max(), V["per_seed_lower_bound_min"],
         V["per_seed_lower_bound_max"], int((per_seed_lb > 0).sum()), V["seeds"]))
print("  data source      %s" % ARC_SOURCE)

ARC-TGI, mean over 20 seeds, scored on the sealed panel
  L_Q(p_ref) = 2.397895   L_Q(p_0) = 2.484117   L_Q(p_T) = 1.747562

  1. terminal level        B = +0.6503 nats
  2. signed gain           G = +0.7366 nats
  3. slack             Gamma = 0.112841 nats   [route c]
  4. lower bound   G - Gamma = +0.6237 nats
  5. ratio         G / Gamma = 6.53
     -> improved by at least 0.6237 nats on the declared target
reconciles: 5000 steps, sum +0.736554907 against endpoint difference +0.736554907, residual +1.11e-16

VALIDATION PASSED against the published ARC-TGI values.
  Delta            0.056421   (target 0.056421)
  Gamma            0.112841   (target 0.112841)
  mean L_0         2.4841     (target 2.4841)
  mean L_T         1.7476     (target 1.7476)
  mean G           0.7366     (target 0.7366)
  lower bound      0.6237     (target 0.6237)
  ratio            6.53       (target 6.53)
  per-seed lb      0.5266 to 0.6981 (target 0.5266 to 0.6981), positive 20/20
  data source      experi

In [13]:
arc_report = audit_cp_report(
    loss_ref_arc, loss_0_arc, loss_T_arc, arc_slack,
    target_version="arc-tgi/sealed-audit-panel/v1",
    reference_version="uniform-over-11-colour-classes/v1",
    masking_policy_version="arc-tgi/content-cells-pad-excluded/v1",
    audit_unit=("one ARC-TGI panel item; loss is probability-floored cross-entropy at floor 0.01, "
                "averaged over the content cells of that item"),
    n_independent_units=ARC_N,
    R=ARC_R,
    commitment_record={
        "state_commitment": evidence_digest(loss_ref_arc, loss_0_arc, loss_T_arc),
        "committed_at": "archived run, 20 seeds, 5000 training steps per seed",
        "challenge_drawn_after_commitment": True,
        "panel_sealed_before_training": True,
        "note": ("the public artifact ships endpoint losses rather than checkpoint hashes, so the "
                 "commitment recorded here is the digest of the endpoint evidence"),
    },
    component_notes=[
        "scored on the sealed panel; the per-step scheduler reward used a separate 512-example probe",
        "per-seed lower bound ranges %.4f to %.4f, positive in %d of %d seeds"
        % (per_seed_lb.min(), per_seed_lb.max(), int((per_seed_lb > 0).sum()), per_seed_lb.size),
        "data source: %s" % ARC_SOURCE,
    ],
)
print(json.dumps(arc_report, indent=2))

{
  "unit": "nats",
  "terminal_level": 0.650333,
  "signed_gain": 0.736555,
  "lower_bound_on_improvement": 0.623714,
  "discrimination_ratio": 6.5274,
  "target_version": "arc-tgi/sealed-audit-panel/v1",
  "reference_version": "uniform-over-11-colour-classes/v1",
  "masking_policy_version": "arc-tgi/content-cells-pad-excluded/v1",
  "audit_unit": "one ARC-TGI panel item; loss is probability-floored cross-entropy at floor 0.01, averaged over the content cells of that item",
  "n_independent_units": 12288,
  "slack": 0.112841,
  "slack_route": "c",
  "slack_route_label": "fixed panel, declared candidate set",
  "slack_parameters": {
    "n": 12288,
    "R": 4.605170186,
    "delta": 0.05,
    "N": 1.0,
    "Delta": 0.05642055019510268,
    "panel": "ARC-TGI sealed audit panel"
  },
  "transcript_bits": null,
  "commitment_record": {
    "state_commitment": "sha256:6b8b4e6a5efe551e31292f0ba8ddb9976ba4faa79e6e9414abeb0dd7f047d625",
    "committed_at": "archived run, 20 seeds, 5000 traini

## 10. The guardrail in action: the same run, the other panel

The same ARC-TGI run also has a second panel. The per-step scheduler reward was computed on a
separate 512-example probe, reused at every one of the 5,000 rounds, and its scores steered the
scheduler. A gain measured there is not a fixed-model measurement. Route (c) refuses it; route (d)
prices it against the bits released.


In [14]:
steering_probe = declare_panel(
    "ARC-TGI 512-example scheduler probe",
    reward=True,                     # its scores were the per-step scheduler reward
    early_stopping=False,
    checkpoint_selection=False,
    hyperparameter_selection=False,
    data_selection=True,             # the scheduler chose which task family to train on next
    transcript_bits=5000 * 32.0,     # one float32 score released at each of 5000 rounds
    note="reused at every round; scores steered the training scheduler",
)
print(steering_probe.describe())
print()

try:
    compute_slack("c", provenance=steering_probe, n=512, R=ARC_R, delta=ARC_DELTA, N=1)
    raise AssertionError("route (c) should have been refused")
except PanelSteeredError as exc:
    print(exc)

probe_slack = compute_slack("d", provenance=steering_probe, n=512, R=ARC_R, delta=ARC_DELTA)
print()
print("route (d) at n = 512, B = %g bits:" % steering_probe.transcript_bits)
print("  " + str(probe_slack))


def transcript_budget_for_gain(*, gain: float, n: int, R: float, delta: float) -> float:
    """Largest transcript budget B, in bits, at which route (d) still leaves a positive bound."""
    lhs = n * (gain / R) ** 2 / 2.0 - math.log(1.0 / delta)
    return lhs / math.log(2.0)


print()
print("  transcript budget B (bits)      slack Gamma (nats)")
for B in (1, 4, 8, 64, 512, 4096, 160000):
    s = compute_slack("d", provenance=steering_probe, n=512, R=ARC_R, delta=ARC_DELTA,
                      transcript_bits=float(B))
    verdict = "positive bound" if arc_five.signed_gain > s.gamma else "no positive bound"
    print("  %-30d %8.4f    %s" % (B, s.gamma, verdict))

B_max = transcript_budget_for_gain(gain=arc_five.signed_gain, n=512, R=ARC_R, delta=ARC_DELTA)
print()
print("At n = 512 and a gain of %.4f nats, route (d) leaves a positive lower bound only up to"
      % arc_five.signed_gain)
print("B = %.2f bits released across the whole run. The archived probe released about %g bits, so"
      % (B_max, steering_probe.transcript_bits))
print("it supports no lower bound. The same model and the same training run therefore support a")
print("lower bound on the sealed panel and none at all on the probe.")
assert probe_slack.gamma > arc_five.signed_gain

panel 'ARC-TGI 512-example scheduler probe': steered training through reward, data_selection; declared transcript 160000 bits

Route (c) is not available for panel 'ARC-TGI 512-example scheduler probe'.

The panel produced signal that influenced training through: reward, data_selection.
A panel whose scores steered training is not a fixed-model measurement: the terminal state was
chosen with knowledge of the panel, so its empirical loss there is not an unbiased read of L_Q and
the uniform-deviation event that route (c) prices does not cover it.

Use route (d) with an honest bit count of the audit-derived transcript exposed to training:

    compute_slack("d", provenance=prov, n=512, R=4.605170186, delta=0.05, transcript_bits=B)

Count every score, threshold, selected example, timing signal, stopping change and scheduler change
caused by audit feedback. If the released bits cannot be counted, the panel does not support a lower
bound at all.

route (d) at n = 512, B = 160000 bits:
  Gamm

## 11. Optional: producing the loss vectors from a torch model

The core above is numpy-only and never imports torch. This section is a recipe for one common way of
producing the three vectors it wants. The function is defined and not called, and its import sits
inside the body, so this notebook runs with or without torch installed.

Two things to get right. Score the **same** panel in the **same** order under all three quotes, and
apply the probability floor before taking the logarithm so the losses respect the declared range `R`.
Reduce to one value per independent audit unit: if the model emits per-token or per-cell losses,
average them **within** each unit and return one number per unit. Do not return the flattened
per-token vector.


In [15]:
def per_unit_losses_from_torch(model, batches, *, floor: float = 0.01, device: str = "cpu"):
    """OPTIONAL RECIPE, not part of the core and not called by this notebook.

    Returns a 1-D numpy vector of length n_units holding -log p(y|x) under `model`, one entry per
    independent audit unit, floored at `floor` so the loss respects R = -log(floor).

    `batches` yields (x, y, unit_id) where unit_id assigns every scored position to its audit unit
    (its example, document or episode). Positions sharing a unit_id are averaged together, which is
    what keeps n the count of independent units rather than of tokens or cells.
    """
    import torch                       # local import: the core never depends on torch
    import torch.nn.functional as F

    sums: Dict[int, float] = {}
    counts: Dict[int, int] = {}
    model.eval().to(device)
    with torch.no_grad():
        for x, y, unit_id in batches:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            probs = logits.softmax(dim=-1)
            probs = (1.0 - floor) * probs + floor / probs.shape[-1]     # full-support quote
            p_y = probs.gather(-1, y.unsqueeze(-1)).squeeze(-1)
            nll = (-p_y.clamp_min(floor).log()).reshape(-1).cpu().numpy()
            uid = np.asarray(unit_id).reshape(-1)
            for u, v in zip(uid.tolist(), nll.tolist()):
                sums[u] = sums.get(u, 0.0) + v
                counts[u] = counts.get(u, 0) + 1

    units = sorted(sums)
    return np.array([sums[u] / counts[u] for u in units], dtype=np.float64)


print("per_unit_losses_from_torch is defined and not called.")
print("torch imported by this notebook:", "torch" in sys.modules)
print()
print("Usage sketch:")
print("  loss_ref = per_unit_losses_from_torch(reference_model, panel_batches())")
print("  loss_0   = per_unit_losses_from_torch(initial_model,   panel_batches())")
print("  loss_T   = per_unit_losses_from_torch(terminal_model,  panel_batches())")
print("  slack    = compute_slack('c', provenance=panel, n=len(loss_0), R=-math.log(0.01),")
print("                           delta=0.05, N=1)")
print("  report   = audit_cp_report(loss_ref, loss_0, loss_T, slack, ...)")

per_unit_losses_from_torch is defined and not called.
torch imported by this notebook: False

Usage sketch:
  loss_ref = per_unit_losses_from_torch(reference_model, panel_batches())
  loss_0   = per_unit_losses_from_torch(initial_model,   panel_batches())
  loss_T   = per_unit_losses_from_torch(terminal_model,  panel_batches())
  slack    = compute_slack('c', provenance=panel, n=len(loss_0), R=-math.log(0.01),
                           delta=0.05, N=1)
  report   = audit_cp_report(loss_ref, loss_0, loss_T, slack, ...)


## 12. What the report does not say

The bound is restricted to the declared probabilistic target, a bounded or verifier-controlled loss,
authenticated causal order, and audit-derived channels that are actually included in the transcript.
Target misspecification, omitted side channels, alteration of the masking policy, correlated verifier
compromise, and objectives with no predictive encoding all sit outside it.

A gain measured under target version `v` is a statement about `Q_v`. When the target changes, close
the ledger and open the next one under the new version; a bridge state scored under both versions
separates the loss shift caused by the revision from progress under either target, in the same unit.

The route (c) and route (d) terms are conservative by construction. A negative or small result at a
given panel size is a statement about resolution, not about the system.
